# Module 3 Worksheet — RAG: Chunking, Retrieval, Advanced Techniques
**Corrected in this version:** `rag_pure_python.py` itself has been patched — `SimpleVectorStore`/`generate_answer` now use the corrected wrapper internally, so no changes needed here beyond the query rewriting / HyDE calls below.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

In [ ]:
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

## 1. Reproducing the teaser problem: bad retrieval -> confident wrong answer

In [ ]:
bad_store = SimpleVectorStore()
bad_store.add(["The weather today is sunny.", "Cats are popular pets.", "Pizza originated in Italy."])

question = "What is MCP?"
bad_chunks = [t for t, s in bad_store.search(question, k=2)]
bad_answer = generate_answer(question, bad_chunks)
print("Retrieved (bad):", bad_chunks)
print("Answer:", bad_answer)
print("\n-> Does the model admit it doesn't know, or does it hallucinate from general knowledge?")

## 2. The fix: tighter generation prompt + good retrieval

In [ ]:
good_store = SimpleVectorStore()
good_store.add(["MCP standardizes how LLMs call external tools through a client-server interface."])

good_chunks = [t for t, s in good_store.search(question, k=1)]
good_answer = generate_answer(question, good_chunks)
print("Retrieved (good):", good_chunks)
print("Answer:", good_answer)

## 3. Query rewriting

In [ ]:
vague_question = "how do u stop the loop thing"

rewritten = ask(
    "Rewrite the user's question into a precise, well-formed search query. Reply with only the rewritten query.",
    vague_question, max_tokens=50,
)
print("Original:", vague_question)
print("Rewritten:", rewritten)

## 4. HyDE (Hypothetical Document Embeddings)

In [ ]:
hypothetical_doc = ask(
    "Write a short, plausible-sounding paragraph that might answer this question, even if you're not certain it's correct.",
    "How do you implement a stopping condition in an agentic loop?", max_tokens=150,
)
print("Hypothetical doc:", hypothetical_doc)

store = SimpleVectorStore()
store.add([
    "Agent loops typically stop when the model emits a final_answer action or hit max_iterations.",
    "Pizza dough needs to rest for at least 30 minutes before stretching.",
])
print("\nSearch with raw question:", store.search("how do u stop the loop thing", k=1))
print("Search with hypothetical doc:", store.search(hypothetical_doc, k=1))

## Teaser exercise
Try multi-query: generate 3 rephrasings of the vague question with `ask()`, retrieve for each, and merge/deduplicate the results. Does the merged set cover more of what a single query would have missed?